# 05b · RAGAS Evaluation — `scale_1K` (OpenAI judge)
*Reads `eval_runs.parquet` from 05. Does NOT re-run generation.*

In [ ]:
import os, sys, json, random, ast
import numpy as np
import pandas as pd

SCALE_LABEL = "scale_1K"
N_PAPERS    = 1000

BASE_DIR    = os.path.abspath("../..")
RESULTS_DIR = os.path.join(BASE_DIR, "4_results", SCALE_LABEL)
print(f"Results dir: {RESULTS_DIR}")

In [ ]:
from dotenv import load_dotenv
load_dotenv(os.path.join(BASE_DIR, ".env"))

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY not found in .env"
print("OpenAI key loaded ✓")

JUDGE_MODEL = "gpt-4o-mini"
print(f"Judge model : {JUDGE_MODEL}")

In [ ]:
# Quick API test
from langchain_openai import ChatOpenAI
test_llm = ChatOpenAI(model=JUDGE_MODEL, api_key=OPENAI_API_KEY, max_tokens=10)
resp = test_llm.invoke("Say hello in one word.")
print(f"API test OK: {resp.content}")

In [ ]:
import subprocess
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ragas==0.2.6", "langchain-openai",
    "langchain-huggingface", "sentence-transformers",
])
print("Dependencies ready ✓")

In [ ]:
parquet_path = os.path.join(RESULTS_DIR, "eval_runs.parquet")
assert os.path.exists(parquet_path), f"Not found: {parquet_path}\nRun 05_generation_ragas.ipynb first."

df_eval = pd.read_parquet(parquet_path)
print(f"Loaded {len(df_eval)} rows")

error_mask = df_eval["answer"].str.startswith("ERROR:")
if error_mask.any():
    print(f"Dropping {error_mask.sum()} error rows")
    df_eval = df_eval[~error_mask].reset_index(drop=True)

assert len(df_eval) >= 40, f"Only {len(df_eval)} valid answers — re-run 05 first."
print(f"Valid answers: {len(df_eval)}")

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

def to_list(val):
    if isinstance(val, list): return [str(v) for v in val]
    if isinstance(val, np.ndarray): return val.tolist()
    if isinstance(val, str):
        try: return ast.literal_eval(val)
        except: return [val]
    return list(val)

contexts_fixed = [to_list(c) for c in df_eval["contexts"].tolist()]

ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(model=JUDGE_MODEL, api_key=OPENAI_API_KEY, temperature=0, max_tokens=512)
)
ragas_emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)
run_config = RunConfig(timeout=120, max_retries=5, max_workers=2)

ragas_ds = Dataset.from_dict({
    "question":     df_eval["question"].tolist(),
    "answer":       df_eval["answer"].tolist(),
    "contexts":     contexts_fixed,
    "ground_truth": df_eval["ground_truth"].tolist(),
})

print(f"Running RAGAS on {len(ragas_ds)} samples (judge={JUDGE_MODEL})...")
result = evaluate(
    ragas_ds,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm, embeddings=ragas_emb, run_config=run_config,
)

In [9]:
print("\n=== RAGAS RESULTS ===")
print(result) # Display the summary

# 1. Save detailed scores (individual rows) to CSV
out_csv = os.path.join(RESULTS_DIR, "ragas_scores.csv")
df_results = result.to_pandas()
df_results.to_csv(out_csv, index=False)
print(f"\nSaved per-query scores : {out_csv}")

# 2. Extract aggregate scores (averages) for the JSON file
# We calculate the mean directly from the numeric columns in the dataframe
agg_scores = df_results.select_dtypes(include=['number']).mean().to_dict()

# 3. Save aggregate scores to JSON
agg_path = os.path.join(RESULTS_DIR, "ragas_agg.json")
with open(agg_path, "w") as f:
    import json
    json.dump({
        "scale": SCALE_LABEL, 
        "judge_model": JUDGE_MODEL, 
        **agg_scores
    }, f, indent=2)
print(f"Saved aggregate scores  : {agg_path}")



=== RAGAS RESULTS ===
{'faithfulness': 0.9169, 'answer_relevancy': 0.6796, 'context_precision': 0.0950, 'context_recall': 0.2600}

Saved per-query scores : d:\SciRet-Scientific-Information-Made-Easy\Sciret2\4_results\scale_1K\ragas_scores.csv
Saved aggregate scores  : d:\SciRet-Scientific-Information-Made-Easy\Sciret2\4_results\scale_1K\ragas_agg.json
